# Avito: обнаружение автоматизированных cookie

Ноутбук строит признаки по событиям, обучает ансамбль из трех CatBoost-моделей и записывает `submission.csv`. Целевая переменная используется только для обучения; в признаки попадают лишь события внутри окна каждой cookie.

Помимо частот и разнообразия событий, признаки описывают регулярность интервалов, паузы между сессиями, последовательности типов событий, концентрацию на объявлениях, активные минуты, поисковое поведение, движения указателя и структуру User-Agent. Итоговый `score` — среднее процентильных рангов моделей с seed `42`, `17` и `2025`; порог к нему не применяется.

## Данные и запуск

Положите исходные файлы в папку `data/` рядом с ноутбуком: `train.csv`, `test.csv`, `events.csv.gz`. Установите зависимости `numpy`, `pandas` и `catboost` (версии закреплены в `requirements.txt`), затем выполните ячейки сверху вниз. Финальная ячейка создаст `submission.csv` с колонками `cookie_id,score`.

In [2]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.4 MB/s eta 0:00:00:00:0100:01m


In [5]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

# Общие поля и фиксированные seed ансамбля.
META_COLUMNS = [
    "cookie_id",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]
EVENT_COLUMNS = [
    "cookie_id",
    "event_ts",
    "eid",
    "event_name",
    "platform",
    "user_agent",
    "item_id",
    "item_category",
    "item_location",
    "seller_type",
    "search_query",
    "search_page",
    "pointer_x",
    "pointer_y",
]
CATEGORICAL_FEATURES = [
    "ua_mode",
    "browser_mode",
    "os_mode",
    "platform_mode",
    "event_mode",
    "category_mode",
    "location_mode",
    "seller_mode",
]
EXCLUDED_MODEL_FEATURES = {"duplicate_events"}
MODEL_SEEDS = (42, 17, 2025)
BOT_UA_PATTERN = re.compile(
    r"bot|crawler|spider|headless|selenium|playwright|curl|python-requests", re.I
)


def _entropy(values: pd.Series) -> float:
    """Энтропия распределения значений в одной группе."""
    frequencies = values.value_counts(normalize=True, dropna=False).to_numpy()
    return float(-(frequencies * np.log(frequencies)).sum()) if len(frequencies) else 0.0


def _group_entropy(events: pd.DataFrame, column: str, feature_name: str) -> pd.Series:
    """Считает энтропию указанного события для каждой cookie."""
    counts = events.groupby(["cookie_id", column], observed=True, dropna=False).size()
    proportions = counts.div(counts.groupby(level=0, sort=False).transform("sum"))
    return (-(proportions * np.log(proportions)).groupby(level=0, sort=False).sum()).rename(
        feature_name
    )


def _group_mode(events: pd.DataFrame, column: str, feature_name: str) -> pd.Series:
    """Выбирает самое частое значение; ничьи разрешаются лексикографически."""
    counts = events.groupby(["cookie_id", column], observed=True, dropna=False).size()
    ranked = counts.rename("count").reset_index()
    ranked = ranked.sort_values(
        ["cookie_id", "count", column],
        ascending=[True, False, True],
        kind="mergesort",
    )
    return ranked.drop_duplicates("cookie_id").set_index("cookie_id")[column].rename(
        feature_name
    )


def _parse_user_agent(events: pd.DataFrame) -> pd.DataFrame:
    """Извлекает браузер, ОС, версию браузера и признак автоматизации."""
    ua = events["user_agent"].fillna("").astype(str)
    ua_lower = ua.str.lower()
    browser_rules = [
        (r"edg/|edge/", "edge"),
        (r"yabrowser/", "yandex"),
        (r"opr/|opera", "opera"),
        (r"chrome/|crios/", "chrome"),
        (r"firefox/|fxios/", "firefox"),
        (r"safari/", "safari"),
    ]
    browser = pd.Series("other", index=events.index, dtype="object")
    for pattern, name in browser_rules:
        browser.loc[ua_lower.str.contains(pattern, regex=True)] = name

    os_rules = [
        (r"windows", "windows"),
        (r"android", "android"),
        (r"iphone|ipad|ipod|ios", "ios"),
        (r"macintosh|mac os", "macos"),
        (r"linux|x11", "linux"),
    ]
    operating_system = pd.Series("other", index=events.index, dtype="object")
    for pattern, name in os_rules:
        operating_system.loc[ua_lower.str.contains(pattern, regex=True)] = name

    browser_version = ua.str.extract(
        r"(?:Edg|Edge|YaBrowser|Chrome|CriOS|Firefox|FxiOS|Version)/([\d.]+)",
        expand=False,
    ).str.split(".").str[0]

    return pd.DataFrame(
        {
            "ua_mode": ua.replace("", "__missing__"),
            "browser": browser,
            "operating_system": operating_system,
            "browser_major": browser_version.fillna("__missing__"),
            "ua_automation_hint": ua.str.contains(BOT_UA_PATTERN).astype("int8"),
        },
        index=events.index,
    )

In [6]:
def build_features(meta: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Строит поведенческие агрегаты только по событиям внутри окна cookie."""
    missing_meta = set(META_COLUMNS).difference(meta.columns)
    missing_events = set(EVENT_COLUMNS).difference(events.columns)
    if missing_meta:
        raise ValueError(f"Missing metadata columns: {sorted(missing_meta)}")
    if missing_events:
        raise ValueError(f"Missing event columns: {sorted(missing_events)}")
    if meta["cookie_id"].duplicated().any():
        raise ValueError("Metadata must contain exactly one row per cookie_id")

    cookies = meta[META_COLUMNS].copy()
    for column in ("cookie_created_at", "window_start_ts", "window_end_ts"):
        cookies[column] = pd.to_datetime(cookies[column], errors="coerce")

    selected_events = events[EVENT_COLUMNS].copy()
    selected_events["event_ts"] = pd.to_datetime(selected_events["event_ts"], errors="coerce")
    selected_events = selected_events.dropna(subset=["event_ts"])
    selected_events = selected_events.merge(
        cookies[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    # Правая граница окна не включается: [window_start_ts, window_end_ts).
    in_window = selected_events["event_ts"].ge(selected_events["window_start_ts"])
    in_window &= selected_events["event_ts"].lt(selected_events["window_end_ts"])
    ev = selected_events.loc[in_window].copy()

    categorical_source = [
        "platform",
        "event_name",
        "item_category",
        "item_location",
        "seller_type",
    ]
    for column in categorical_source:
        ev[column] = ev[column].fillna("__missing__").astype(str).str.strip().str.lower()
        ev[column] = ev[column].replace("", "__missing__")
    ev["user_agent"] = ev["user_agent"].fillna("").astype(str).str.strip()
    ev["search_query"] = ev["search_query"].fillna("").astype(str).str.strip().str.lower()
    ev["search_page"] = pd.to_numeric(ev["search_page"], errors="coerce")
    ev["pointer_x"] = pd.to_numeric(ev["pointer_x"], errors="coerce")
    ev["pointer_y"] = pd.to_numeric(ev["pointer_y"], errors="coerce")
    ev = ev.join(_parse_user_agent(ev))

    # Используем агрегаты по User-Agent вместо подачи его длины/версии
    # как технических характеристик конкретного устройства.
    ev["ua_length"] = ev["user_agent"].str.len()
    ev["browser_major_numeric"] = pd.to_numeric(ev["browser_major"], errors="coerce")
    ev["search_query_length"] = ev["search_query"].str.len()

    # Удаляем полные дубликаты из поведенческих агрегатов, но считаем их отдельно.
    ev["is_duplicate"] = ev.duplicated(subset=EVENT_COLUMNS, keep="first").astype("int8")
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort")
    duplicate_counts = ev.groupby("cookie_id", sort=False)["is_duplicate"].sum().rename(
        "duplicate_events"
    )
    ev = ev.drop_duplicates(subset=EVENT_COLUMNS, keep="first")

    # Последовательность событий показывает повторения и типичные переходы.
    previous_event = ev.groupby("cookie_id", sort=False)["event_name"].shift(1)
    has_previous_event = previous_event.notna()
    ev["same_event_as_previous"] = previous_event.eq(ev["event_name"]) & has_previous_event
    transition_events = ev.loc[has_previous_event, ["cookie_id", "event_name"]].copy()
    transition_events["event_transition"] = (
        previous_event.loc[has_previous_event].astype(str)
        + " -> "
        + transition_events["event_name"].astype(str)
    )

    grouped = ev.groupby("cookie_id", sort=False)
    features = grouped.agg(
        n_events=("event_ts", "size"),
        n_event_types=("event_name", "nunique"),
        n_items=("item_id", "nunique"),
        n_categories=("item_category", "nunique"),
        n_locations=("item_location", "nunique"),
        n_seller_types=("seller_type", "nunique"),
        n_platforms=("platform", "nunique"),
        n_user_agents=("user_agent", "nunique"),
        ua_length_mean=("ua_length", "mean"),
        ua_length_std=("ua_length", "std"),
        browser_major_nunique=("browser_major", "nunique"),
        browser_major_mean=("browser_major_numeric", "mean"),
        browser_major_std=("browser_major_numeric", "std"),
        first_event_ts=("event_ts", "min"),
        last_event_ts=("event_ts", "max"),
        pointer_events=("pointer_x", "count"),
        search_page_mean=("search_page", "mean"),
        search_page_max=("search_page", "max"),
        search_page_std=("search_page", "std"),
        search_page_nunique=("search_page", "nunique"),
        search_query_length_mean=("search_query_length", "mean"),
        search_query_length_max=("search_query_length", "max"),
        pointer_x_nunique=("pointer_x", "nunique"),
        pointer_y_nunique=("pointer_y", "nunique"),
        ua_automation_hint=("ua_automation_hint", "max"),
        same_event_repeat_share=("same_event_as_previous", "mean"),
    )
    mode_sources = {
        "ua_mode": "ua_mode",
        "browser_mode": "browser",
        "os_mode": "operating_system",
        "platform_mode": "platform",
        "event_mode": "event_name",
        "category_mode": "item_category",
        "location_mode": "item_location",
        "seller_mode": "seller_type",
    }
    for feature_name, source_column in mode_sources.items():
        features = features.join(_group_mode(ev, source_column, feature_name))

    if not transition_events.empty:
        transition_features = transition_events.groupby("cookie_id").agg(
            n_event_transitions=("event_transition", "size"),
            n_unique_event_transitions=("event_transition", "nunique"),
        )
        features = features.join(transition_features)
        features = features.join(
            _group_entropy(transition_events, "event_transition", "event_transition_entropy")
        )

    searched = ev.loc[ev["search_query"].ne("")].groupby("cookie_id")["search_query"].nunique()
    features["n_search_queries"] = searched
    features = features.join(duplicate_counts)
    features["event_span_seconds"] = (
        features["last_event_ts"] - features["first_event_ts"]
    ).dt.total_seconds()

    # Entropy и доля самого частого объявления отражают концентрацию просмотра.
    features = features.join(_group_entropy(ev, "item_id", "item_entropy"))
    item_counts = ev.dropna(subset=["item_id"]).groupby(["cookie_id", "item_id"]).size()
    if not item_counts.empty:
        item_total = item_counts.groupby(level=0).sum()
        item_top_share = (item_counts.groupby(level=0).max() / item_total).rename(
            "top_item_event_share"
        )
        features = features.join(item_top_share)

    # Интервалы между событиями описывают регулярность и разрывы между сессиями.
    gap_frame = ev[["cookie_id"]].copy()
    gap_frame["gap_seconds"] = grouped["event_ts"].diff().dt.total_seconds()
    valid_gaps = gap_frame.dropna(subset=["gap_seconds"])
    if not valid_gaps.empty:
        gap_features = valid_gaps.groupby("cookie_id")["gap_seconds"].agg(
            gap_mean_seconds="mean",
            gap_median_seconds="median",
            gap_std_seconds="std",
            gap_min_seconds="min",
            gap_p10_seconds=lambda values: values.quantile(0.1),
            gap_p90_seconds=lambda values: values.quantile(0.9),
        )
        gap_features["gap_cv"] = gap_features["gap_std_seconds"] / gap_features[
            "gap_mean_seconds"
        ].replace(0, np.nan)
        gap_grouped = valid_gaps.groupby("cookie_id")["gap_seconds"]
        gap_features["gap_under_0_5s_share"] = gap_grouped.apply(
            lambda values: values.le(0.5).mean()
        )
        gap_features["gap_under_2s_share"] = gap_grouped.apply(
            lambda values: values.le(2).mean()
        )
        gap_features["gap_under_10s_share"] = gap_grouped.apply(
            lambda values: values.le(10).mean()
        )
        gap_features["gap_over_30m_share"] = gap_grouped.apply(
            lambda values: values.gt(1800).mean()
        )
        gap_features["session_count"] = gap_grouped.apply(
            lambda values: 1 + values.gt(1800).sum()
        )
        features = features.join(gap_features)

    for column, feature_name in [
        ("event_name", "event_entropy"),
        ("item_category", "category_entropy"),
        ("item_location", "location_entropy"),
        ("platform", "platform_entropy"),
        ("seller_type", "seller_entropy"),
        ("browser", "browser_entropy"),
    ]:
        features = features.join(_group_entropy(ev, column, feature_name))

    event_counts = pd.crosstab(ev["cookie_id"], ev["event_name"]).add_prefix("event_count_")
    features = features.join(event_counts)
    if not ev.empty:
        hour_counts = pd.crosstab(ev["cookie_id"], ev["event_ts"].dt.hour)
        active_hours = (hour_counts > 0).sum(axis=1).rename("active_hours")
        night_count = hour_counts.reindex(columns=range(0, 6), fill_value=0).sum(axis=1)
        night_share = (night_count / hour_counts.sum(axis=1)).rename("night_event_share")
        hour_entropy = hour_counts.apply(_entropy, axis=1).rename("hour_entropy")
        active_minutes = ev.assign(active_minute=ev["event_ts"].dt.floor("min")).groupby(
            "cookie_id"
        )["active_minute"].nunique().rename("active_minutes")
        features = features.join([active_hours, night_share, hour_entropy, active_minutes])

        pointer_events = ev.dropna(subset=["pointer_x", "pointer_y"])
        if not pointer_events.empty:
            pointer_events = pointer_events.copy()
            pointer_events["step_distance"] = np.hypot(
                pointer_events.groupby("cookie_id")["pointer_x"].diff(),
                pointer_events.groupby("cookie_id")["pointer_y"].diff(),
            )
            pointer_features = pointer_events.groupby("cookie_id").agg(
                pointer_valid_events=("step_distance", "count"),
                pointer_path_length=("step_distance", "sum"),
                pointer_step_mean=("step_distance", "mean"),
                pointer_step_std=("step_distance", "std"),
                pointer_step_max=("step_distance", "max"),
                pointer_stationary_share=(
                    "step_distance",
                    lambda values: values.eq(0).mean(),
                ),
            )
            features = features.join(pointer_features)

    features = features.reset_index()
    features = cookies.merge(features, on="cookie_id", how="left", validate="one_to_one")

    duration = (features["window_end_ts"] - features["window_start_ts"]).dt.total_seconds()
    age = (features["window_start_ts"] - features["cookie_created_at"]).dt.total_seconds()
    features["window_duration_hours"] = duration / 3600
    features["cookie_age_days"] = age / 86400
    features["cookie_created_hour"] = features["cookie_created_at"].dt.hour
    features["window_start_hour"] = features["window_start_ts"].dt.hour
    features["window_start_weekday"] = features["window_start_ts"].dt.dayofweek
    features["first_event_offset_seconds"] = (
        features["first_event_ts"] - features["window_start_ts"]
    ).dt.total_seconds()
    features["last_event_offset_seconds"] = (
        features["window_end_ts"] - features["last_event_ts"]
    ).dt.total_seconds()
    features["events_per_active_hour"] = features["n_events"] / features[
        "active_hours"
    ].replace(0, np.nan)
    features["events_per_active_minute"] = features["n_events"] / features[
        "active_minutes"
    ].replace(0, np.nan)

    event_count_columns = [column for column in features if column.startswith("event_count_")]
    for column in event_count_columns:
        features[f"{column}_share"] = features[column] / features["n_events"].replace(0, np.nan)

    dropped = META_COLUMNS[1:] + ["first_event_ts", "last_event_ts"]
    features = features.drop(columns=dropped)
    for column in CATEGORICAL_FEATURES:
        if column not in features:
            features[column] = "__missing__"
        features[column] = features[column].fillna("__missing__").astype(str)
    numeric_columns = features.select_dtypes(include=[np.number]).columns
    features[numeric_columns] = features[numeric_columns].replace([np.inf, -np.inf], np.nan)
    return features

In [7]:
def make_catboost_model(random_seed: int = 42) -> CatBoostClassifier:
    """Параметры совпадают с решением, на котором получен submission.csv."""
    return CatBoostClassifier(
        iterations=400,
        depth=5,
        learning_rate=0.05,
        l2_leaf_reg=5,
        loss_function="Logloss",
        verbose=False,
        random_seed=random_seed,
        thread_count=4,
        allow_writing_files=False,
    )


def split_feature_columns(features: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Отделяет категориальные признаки для нативной обработки CatBoost."""
    categorical = [column for column in CATEGORICAL_FEATURES if column in features]
    numeric = [
        column
        for column in features.select_dtypes(include=[np.number]).columns
        if column not in EXCLUDED_MODEL_FEATURES
    ]
    return categorical, numeric


def catboost_frame(
    features: pd.DataFrame,
    categorical_features: list[str],
    numeric_features: list[str],
) -> pd.DataFrame:
    """Формирует матрицу модели и безопасно заполняет категории."""
    model_features = features[categorical_features + numeric_features].copy()
    for column in categorical_features:
        model_features[column] = model_features[column].fillna("__missing__").astype(str)
    return model_features


def average_rank(scores: list[np.ndarray]) -> np.ndarray:
    """Усредняет процентильные ранги, снижая зависимость от seed модели."""
    ranks = [pd.Series(values).rank(method="average", pct=True).to_numpy() for values in scores]
    return np.mean(ranks, axis=0)

In [8]:
# Читаем таблицы исходных данных. Из train берется target только для обучения.
data_dir = Path("data")
output_path = Path("submission.csv")

train = pd.read_csv(data_dir / "train.csv", parse_dates=META_COLUMNS[1:])
test = pd.read_csv(data_dir / "test.csv", parse_dates=META_COLUMNS[1:])
events = pd.read_csv(data_dir / "events.csv.gz", parse_dates=["event_ts"])

# Строим признаки одним проходом по общему набору метаданных, затем сохраняем
# исходный порядок строк train/test при разделении матриц.
all_meta = pd.concat(
    [train[META_COLUMNS], test[META_COLUMNS]],
    ignore_index=True,
)
all_features = build_features(all_meta, events)
train_features = train[["cookie_id"]].merge(
    all_features, on="cookie_id", how="left", validate="one_to_one"
).drop(columns="cookie_id")
test_features = test[["cookie_id"]].merge(
    all_features, on="cookie_id", how="left", validate="one_to_one"
).drop(columns="cookie_id")

categorical_features, numeric_features = split_feature_columns(train_features)
train_model_features = catboost_frame(
    train_features, categorical_features, numeric_features
)
test_model_features = catboost_frame(
    test_features, categorical_features, numeric_features
)

# Обучаем три модели на всех размеченных примерах и усредняем их ранги.
test_seed_scores = []
for seed in MODEL_SEEDS:
    model = make_catboost_model(seed)
    model.fit(
        train_model_features,
        train["target"],
        cat_features=categorical_features,
    )
    test_seed_scores.append(model.predict_proba(test_model_features)[:, 1])

submission = pd.DataFrame(
    {
        "cookie_id": test["cookie_id"].to_numpy(),
        "score": average_rank(test_seed_scores),
    }
)

# Проверяем, что в файле ровно один допустимый score на каждую test-cookie.
if len(submission) != len(test) or submission["cookie_id"].duplicated().any():
    raise ValueError("Submission must contain exactly one row per test cookie")
if not submission["score"].between(0, 1).all():
    raise ValueError("Submission contains scores outside [0, 1]")
if set(submission["cookie_id"]) != set(test["cookie_id"]):
    raise ValueError("Submission cookie IDs do not match test.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)
print(f"Сохранено {len(submission)} прогнозов: {output_path}")
submission.head()

Сохранено 4909 прогнозов: submission.csv


,cookie_id,score
0,ck_315fb710a0e371e7,0.009982
1,ck_a76ee3b3e3e522fd,0.909214
2,ck_94c9a4d382689e82,0.771101
3,ck_8eaf9509ad9462a0,0.067767
4,ck_9a88a5a989cb5bc6,0.394921
